# NeuralNav — Retrieval (sentence-transformers + FAISS) and NER (spaCy)

Run on Kaggle (GPU optional here — embedding a small KB is fast even on CPU, but leave GPU on since it's free with your quota).

Uses the `kb.json` auto-derived from the real Bitext dataset in `01_intent_classification.ipynb` (download it from that notebook's output and upload here as a Kaggle Dataset, e.g. `neuralnav-kb`), with an inline fallback if it isn't found.

Outputs: `kb_embeddings.npy`, `kb_index.faiss` — download into your local `models/` folder. The FastAPI backend can either rebuild the index at startup from `data/kb.json` (current default in `ml/retrieval.py`) or load these pre-built files directly.

In [ ]:
!pip install -q sentence-transformers faiss-cpu spacy
!python -m spacy download en_core_web_sm -q

In [ ]:
import os, json

KB_PATH = "/kaggle/input/neuralnav-kb/kb.json"
if not os.path.exists(KB_PATH):
    KB_PATH = "/kaggle/working/kb.json"
    print("NOTE: using fallback — upload your real data/kb.json as a Kaggle dataset for the real run.")
    sample_kb = [
        {"id": "kb_power_1", "intent": "power_issue", "question": "Device won't turn on",
         "answer": "Hold the power button for 10 seconds to force a restart."}
    ]
    json.dump(sample_kb, open(KB_PATH, "w"))

kb = json.load(open(KB_PATH))
len(kb)

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "all-MiniLM-L6-v2"
model = SentenceTransformer(EMBED_MODEL)

texts = [f"{item['question']} {item['answer']}" for item in kb]
embeddings = model.encode(texts, normalize_embeddings=True)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(np.asarray(embeddings, dtype=np.float32))

np.save("/kaggle/working/kb_embeddings.npy", embeddings)
faiss.write_index(index, "/kaggle/working/kb_index.faiss")
print("Saved embeddings + FAISS index.")

In [ ]:
# Sanity check the retrieval quality before downloading
query = "my wifi disconnects randomly"
q_emb = model.encode([query], normalize_embeddings=True)
scores, idxs = index.search(np.asarray(q_emb, dtype=np.float32), 3)
for score, idx in zip(scores[0], idxs[0]):
    print(round(float(score), 3), kb[idx]["question"], "->", kb[idx]["answer"])

## NER sanity check (spaCy + regex for structured entities)

In [ ]:
import re
import spacy

nlp = spacy.load("en_core_web_sm")
ORDER_ID_RE = re.compile(r"\b(?:order\s*#?\s*)?([A-Z]{0,3}\d{5,10})\b", re.IGNORECASE)
ERROR_CODE_RE = re.compile(r"\b(E\d{3}|0x[0-9A-Fa-f]{2,4})\b")

def extract_entities(text):
    order_ids = [m.group(1) for m in ORDER_ID_RE.finditer(text)]
    error_codes = ERROR_CODE_RE.findall(text)
    doc = nlp(text)
    spacy_entities = [{"text": ent.text, "label": ent.label_} for ent in doc.ents]
    return {"order_ids": order_ids, "error_codes": error_codes, "spacy_entities": spacy_entities}

extract_entities("I'm getting error code E101 on order 4471829, shipped from Berlin")